# Premade Model Testing

## Setting up functions

In [9]:
%pip install transformers datasets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.5.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (12 kB)
  Using cached multiprocess-0.70.16-py311-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.3.2-py2.py3-none-any.whl.metadata (3.8 kB)
Using cached dill-0.3.8-py3-none-any.whl (116 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)
Using cached multiprocess-0.70.16-py311-none-any.whl (143 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.9/30.9 MB 73.7 MB/s eta 0:00:0000:0100:01
Using cached xxhash-3.5.0-cp311-cp311-macosx_11_0_arm64.whl (30 kB)
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.3.2-py2.py3-none-any.whl (7.6 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uni

In [10]:
from transformers import pipeline
from datasets import load_dataset
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/betaknight/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [11]:
import wikipedia

wikipedia.set_lang("en")
article_text = wikipedia.page("Pro-life").content

In [12]:
sentences = sent_tokenize(article_text)
print(f"Total sentences: {len(sentences)}")

Total sentences: 55


In [13]:
toxicity_classifier = pipeline("text-classification", model="unitary/unbiased-toxic-roberta")
bias_classifier = pipeline("text-classification", model="newsmediabias/UnBIAS-classifier")

Device set to use mps:0


config.json:   0%|          | 0.00/921 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Device set to use mps:0


In [14]:
bias_lexicon_dataset = load_dataset("mediabiasgroup/bias-lexicon")
biased_words_set = set(bias_lexicon_dataset['train']['word'])

README.md:   0%|          | 0.00/676 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


bias_lexicon.csv:   0%|          | 0.00/32.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2130 [00:00<?, ? examples/s]

In [15]:
results = []

for i, sentence in enumerate(sentences):
    toxicity = toxicity_classifier(sentence)[0]
    bias = bias_classifier(sentence)[0]
    biased_words = [word for word in sentence.lower().split() if word in biased_words_set]

    results.append({
        "sentence": sentence,
        "toxicity_score": toxicity['score'] if toxicity['label'] == 'toxicity' else 0.0,
        "bias_label": bias['label'],
        "bias_score": bias['score'],
        "biased_words": biased_words
    })

In [16]:
import pandas as pd

df_results = pd.DataFrame(results)
df_results.head(10)

,sentence,toxicity_score,bias_label,bias_score,biased_words
0,Pro-Life (born Marvin Thomas Richardson; Augus...,0.000476,Highly Biased,0.804774,[]
1,He lives in the unincorporated community of Le...,0.000582,Slightly Biased,0.583561,[]
2,He has made several unsuccessful runs for poli...,0.009212,Highly Biased,0.549979,[]
3,Pro-Life ran in the 2020 United States House o...,0.000356,Highly Biased,0.623362,[]
4,== Early life and education ==\nBorn Marvin Th...,0.000383,Highly Biased,0.649666,[]
5,He played basketball in high school and attend...,0.000420,Slightly Biased,0.551373,[]
6,He graduated from BYU in 1967 with a degree in...,0.000571,Highly Biased,0.458142,[]
7,Pro-Life has worked as an organic farmer since...,0.000410,Slightly Biased,0.508609,[]
8,"He has previously worked as an accountant, coa...",0.000839,Highly Biased,0.649551,[]
9,== Career ==\n\n\n=== Campaigns ===\nAs Marvin...,0.000440,Highly Biased,0.719653,[]


In [17]:
avg_toxicity = df_results['toxicity_score'].mean()
bias_distribution = df_results['bias_label'].value_counts()
total_biased_words = df_results['biased_words'].apply(len).sum()

print(f"Average Toxicity Score: {avg_toxicity:.4f}")
print("\nBias Label Distribution:")
print(bias_distribution)
print(f"\nTotal Biased Words Found: {total_biased_words}")

Average Toxicity Score: 0.0025

Bias Label Distribution:
bias_label
Highly Biased      24
Slightly Biased    21
Neutral            10
Name: count, dtype: int64

Total Biased Words Found: 1


Great! Let's add some spaCy

In [20]:
%pip install spacy
%python -m spacy download en_core_web_sm

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.3.1 -> 25.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


UsageError: Line magic function `%python` not found (But cell magic `%%python` exists, did you mean that instead?).


In [22]:
import spacy
nlp = spacy.load("en_core_web_sm")

bias_phrases = {
    "factive_verbs": {"admit", "confirm", "prove", "demonstrate", "reveal"},
    "hedges": {"reportedly", "allegedly", "apparently", "may", "might", "seem", "suggests"},
    "intensifiers": {"clearly", "obviously", "undeniably", "very", "highly"},
    "implicatives": {"manage", "fail", "refuse", "hesitate"}
}
combined_lexicon = set().union(*bias_phrases.values())

results = []

for i, sentence in enumerate(sentences):
    toxicity = toxicity_classifier(sentence)[0]
    bias = bias_classifier(sentence)[0]

    doc = nlp(sentence)
    cue_matches = [token.text.lower() for token in doc if token.text.lower() in combined_lexicon]
    biased_words = [word for word in sentence.lower().split() if word in biased_words_set]
    all_biased_words = list(set(biased_words + cue_matches))

    results.append({
        "sentence": sentence,
        "toxicity_score": toxicity['score'] if toxicity['label'] == 'toxicity' else 0.0,
        "bias_label": bias['label'],
        "bias_score": bias['score'],
        "biased_words": all_biased_words
    })

In [ ]:
df_results = pd.DataFrame(results)

avg_toxicity = df_results['toxicity_score'].mean()
bias_distribution = df_results['bias_label'].value_counts()
total_biased_words = df_results['biased_words'].apply(len).sum()

print(f"Average Toxicity Score: {avg_toxicity:.4f}")
print("\nBias Label Distribution:")
print(bias_distribution)
print(f"\nTotal Biased Words Found (from both lexicons): {total_biased_words}")

Average Toxicity Score: 0.0025

Bias Label Distribution:
bias_label
Highly Biased      24
Slightly Biased    21
Neutral            10
Name: count, dtype: int64

Total Biased Words Found (from both lexicons): 3


In [ ]:
toxicity_classifier = pipeline("text-classification", model="unitary/unbiased-toxic-roberta")
bias_classifier = pipeline("text-classification", model="newsmediabias/UnBIAS-classifier")

bias_lexicon = load_dataset("mediabiasgroup/bias-lexicon")
biased_words_set = set(bias_lexicon['train']['word'])

bias_phrases = {
    "factive_verbs": {"admit", "confirm", "prove", "demonstrate", "reveal"},
    "hedges": {"reportedly", "allegedly", "apparently", "may", "might", "seem", "suggests"},
    "intensifiers": {"clearly", "obviously", "undeniably", "very", "highly"},
    "implicatives": {"manage", "fail", "refuse", "hesitate"}
}
combined_lexicon = set().union(*bias_phrases.values())

topics = [
    "Pro-life", "Pro-choice", "Gun control", "Climate change", "Donald Trump",
    "Barack Obama", "Universal healthcare", "Abortion in the United States", "Feminism", "Immigration"
]

summary = []

for topic in topics:
    try:
        real_title = wikipedia.search(topic)[0]
        text = wikipedia.page(real_title).content
        sentences = sent_tokenize(text)

        tox_scores, bias_scores, total_biased_words = [], [], 0

        for sentence in sentences:
            tox = toxicity_classifier(sentence)[0]
            bias = bias_classifier(sentence)[0]
            doc = nlp(sentence)

            cue_hits = [t.text.lower() for t in doc if t.text.lower() in combined_lexicon]
            lexicon_hits = [w for w in sentence.lower().split() if w in biased_words_set]
            all_hits = set(cue_hits + lexicon_hits)
            total_biased_words += len(all_hits)

            tox_scores.append(tox["score"] if tox["label"] == "toxicity" else 0)
            bias_scores.append(bias["score"])

        summary.append({
            "article": real_title,
            "avg_toxicity": round(sum(tox_scores) / len(tox_scores), 4),
            "avg_bias_score": round(sum(bias_scores) / len(bias_scores), 4),
            "total_biased_words": total_biased_words
        })

    except Exception as e:
        summary.append({
            "article": topic,
            "avg_toxicity": "ERROR",
            "avg_bias_score": "ERROR",
            "total_biased_words": f"Error: {str(e)[:40]}"
        })

df_summary = pd.DataFrame(summary)
df_summary

Device set to use mps:0
Device set to use mps:0
Repo card metadata block was not found. Setting CardData to empty.


,article,avg_toxicity,avg_bias_score,total_biased_words
0,Pro-Life (politician),0.0025,0.6147,3
1,Pro-choice and pro-life,0.0006,0.7547,5
2,Gun control,0.0012,0.7331,52
3,Climate change,0.0004,0.6035,29
4,Donald Trump,0.0101,0.6859,86
5,Barack Obama,0.0016,0.6185,49
6,Universal health care,0.0006,0.6348,13
7,Abortion in the United States,0.0047,0.6930,66
8,Feminism,0.0005,0.7878,97
9,Immigration,0.0043,0.7607,57


## Making CSV

In [ ]:
from wikipedia.exceptions import DisambiguationError, PageError

toxicity_classifier = pipeline("text-classification", model="unitary/unbiased-toxic-roberta")
bias_classifier = pipeline("text-classification", model="newsmediabias/UnBIAS-classifier")

bias_lexicon = load_dataset("mediabiasgroup/bias-lexicon")
biased_words_set = set(bias_lexicon['train']['word'])

bias_phrases = {
    "factive_verbs": {"admit", "confirm", "prove", "demonstrate", "reveal"},
    "hedges": {"reportedly", "allegedly", "apparently", "may", "might", "seem", "suggests"},
    "intensifiers": {"clearly", "obviously", "undeniably", "very", "highly"},
    "implicatives": {"manage", "fail", "refuse", "hesitate"}
}
combined_lexicon = set().union(*bias_phrases.values())

topics = [
    "Politics", "Abortion", "Guns", "Healthcare", "Immigration", "Climate", "Gender", "Feminism",
    "Capitalism", "Socialism", "Libertarianism", "Progressivism", "Conservatism", "Liberalism", "Racism",
    "LGBT", "Religion", "Censorship", "Free speech", "Terrorism", "Civil rights", "Police", "Prisons",
    "Taxation", "Social media", "Elections", "Education", "Minimum wage", "Foreign policy",
    "Nationalism", "Patriotism", "Green energy", "Vaccines", "Public health", "Gun violence",
    "Bipartisanship", "Homelessness", "Supreme Court", "Presidency", "Congress", "Senate",
    "House of Representatives", "Media bias", "Fake news", "Propaganda", "Fact-checking",
    "Journalism", "Freedom", "Discrimination", "Protests", "Strikes", "Labor unions", "Corporations",
    "Big tech", "Economy", "Stock market", "Recession", "Surveillance", "Privacy", "Whistleblower",
    "Military", "Defense spending", "Sexual harassment", "Critical race theory", "School choice",
    "Standardized testing", "Cyberbullying", "AI ethics", "Facial recognition", "Climate crisis",
    "Carbon emissions", "Nuclear weapons", "Genocide", "Refugees", "Asylum seekers",
    "DACA", "Affordable housing", "Rent control", "Opioid crisis", "Marijuana legalization",
    "Gerrymandering", "Campaign finance", "Lobbying"
]

summary = []

for i, topic in enumerate(topics):
    try:
        print(f"Processing {i+1}/{len(topics)}: {topic}")
        real_title = wikipedia.search(topic)[0]
        page = wikipedia.page(real_title, auto_suggest=False)
        text = page.content[:10000]
        sentences = sent_tokenize(text)

        tox_scores, bias_scores, total_biased_words = [], [], 0

        for sentence in sentences:
            try:
                tox = toxicity_classifier(sentence)[0]
                bias = bias_classifier(sentence)[0]
                doc = nlp(sentence)

                cue_hits = [t.text.lower() for t in doc if t.text.lower() in combined_lexicon]
                lexicon_hits = [w for w in sentence.lower().split() if w in biased_words_set]
                all_hits = set(cue_hits + lexicon_hits)
                total_biased_words += len(all_hits)

                tox_scores.append(tox["score"] if tox["label"] == "toxicity" else 0)
                bias_scores.append(bias["score"])
            except:
                continue

        if not bias_scores:
            raise ValueError("No valid scores found.")

        avg_tox = round(sum(tox_scores) / len(tox_scores), 4)
        avg_bias = round(sum(bias_scores) / len(bias_scores), 4)
        is_biased = 1 if avg_bias > 0.65 or total_biased_words > 30 else 0

        cleaned_doc = nlp(text)
        cleaned_text = " ".join([token.lemma_ for token in cleaned_doc if token.is_alpha and not token.is_stop])

        summary.append({
            "article": real_title,
            "article_text_cleaned": cleaned_text,
            "avg_toxicity": avg_tox,
            "avg_bias_score": avg_bias,
            "total_biased_words": total_biased_words,
            "is_biased": is_biased
        })

    except (DisambiguationError, PageError, ValueError) as e:
        print(f"Skipped: {topic} — {str(e)[:80]}")
        continue

df_summary = pd.DataFrame(summary)
df_summary.to_csv("wikipedia_bias_with_text.csv", index=False)
print("✅ Saved: wikipedia_bias_with_text.csv")

Device set to use mps:0
Device set to use mps:0
Repo card metadata block was not found. Setting CardData to empty.


Processing 1/99: Politics
Processing 2/99: Abortion
Processing 3/99: Guns
Processing 4/99: Healthcare
Processing 5/99: Immigration
Processing 6/99: Climate
Processing 7/99: Welfare
Skipped: Welfare → "Welfare" may refer to: 
Well-being
utilitarianism
Value
Utility
Decision utilit
Processing 8/99: Gender
Processing 9/99: Feminism
Processing 10/99: Capitalism
Processing 11/99: Socialism
Processing 12/99: Libertarianism
Processing 13/99: Progressivism
Processing 14/99: Conservatism
Processing 15/99: Liberalism
Processing 16/99: Racism
Processing 17/99: Inequality
Skipped: Inequality → "Inequality" may refer to: 
Inequality (mathematics)
Economic inequality
Income 
Processing 18/99: LGBT
Processing 19/99: Religion
Processing 20/99: Censorship
Processing 21/99: Free speech
Processing 22/99: Terrorism
Processing 23/99: Civil rights
Processing 24/99: Police
Processing 25/99: Prisons
Processing 26/99: Taxation
Processing 27/99: Social media
Processing 28/99: War
Processing 29/99: Voting
Proces